# Setting up a relative binding free energy network

This tutorial gives a step-by-step process to set up a relative binding free energy (RBFE) simulation campaign using OpenFE. This tutorial is designed as an accompaniment to the CLI tutorial found in the same directory as this notebook.

With the CLI, all the steps here were performed by the `openfe plan-rbfe-network` command. However, that command offers little room for customization. Using the Python interface gives us the ability to customize all aspects of how our simulation runs. This tutorial provides a step-by-step Python guide to reproducing the setup done in the CLI tutorial, highlighting areas where the Python interface enables customization.

In [ ]:
import multiprocessing as mp
import os
import re
from pathlib import Path

import MDAnalysis as mda
import openfe
import py3Dmol
from MDAnalysis.analysis import align
from openfe import (
    AlchemicalNetwork,
    ChemicalSystem,
    ProteinComponent,
    SmallMoleculeComponent,
    SolventComponent,
    Transformation,
)
from openfe.protocols.openmm_rfe import RelativeHybridTopologyProtocol
from openfe.protocols.openmm_septop import SepTopProtocol
from openfe.protocols.openmm_utils.charge_generation import bulk_assign_partial_charges
from openfe.protocols.openmm_utils.omm_settings import OpenFFPartialChargeSettings
from openfe.setup.alchemical_network_planner import RBFEAlchemicalNetworkPlanner
from openfe.setup.atom_mapping import KartografAtomMapper, LomapAtomMapper
from openfe.setup.atom_mapping.lomap_scorers import default_lomap_score
from openfe.setup.ligand_network_planning import (
    generate_lomap_network,
    generate_minimal_spanning_network,
    generate_network_from_indices,
    generate_network_from_names,
)
from openfe.utils.atommapping_network_plotting import plot_atommapping_network
from openff.units import unit
from rdkit import Chem
from rdkit.Chem import AllChem, Draw
from rdkit.Chem.Descriptors3D import Asphericity

from mdpp.prep import assign_topology, fix_pdb

In [ ]:
COMPLEX_DIR = Path("pdbs")
SMILES_PATH = Path("ligands_amp_fep.smi")
PROTEIN_CHAIN_ID = "A"
LIGAND_CHAIN_ID = "B"

WORKING_DIR = Path("output")
WORKING_DIR.mkdir(exist_ok=True)
TMP_DIR = Path("tmp")
TMP_DIR.mkdir(exist_ok=True)

PROTEIN_PATH = WORKING_DIR / "protein.pdb"
FIXED_PROTEIN_PATH = WORKING_DIR / "protein_fixed.pdb"
ALIGNED_DIR = WORKING_DIR / "aligned"
ALIGNED_DIR.mkdir(exist_ok=True)
CHARGED_DIR = WORKING_DIR / "charged"
CHARGED_DIR.mkdir(exist_ok=True)
LIGAND_NETWORK_PATH = WORKING_DIR / "ligand_network.graphml"
TRANSFORMATION_DIR = WORKING_DIR / "transformations"
TRANSFORMATION_DIR.mkdir(exist_ok=True)
SEPTOP_TRANSFORMATION_DIR = WORKING_DIR / "septop_transformations"
SEPTOP_TRANSFORMATION_DIR.mkdir(exist_ok=True)

# Simulation parameters
TEMPERATURE = 298.15
PH = 7.0
LAMBDA_WINDOWS = 26
PRODUCTION_LENGTH = 5.0

## Prepare structures from complex PDB files

Each complex PDB file contains a protein and ligand. We select the reference protein as the complex bound to the largest ligand by volume (Asphericity), align all other complexes to it, assign correct bond orders from the SMILES file using `assign_topology`, then fix and protonate the reference protein with `fix_pdb`.

In [ ]:
def _match_ligand(stem: str, smi_d: dict[str, Chem.Mol]) -> str | None:
    """Return the SMILES name that whole-word matches the PDB filename stem."""
    return next(
        (n for n in smi_d if re.search(rf"(?<![A-Za-z\d]){re.escape(n)}(?![A-Za-z\d])", stem)),
        None,
    )


# Build SMILES template dict from .smi file
smi_suppl = Chem.SmilesMolSupplier(str(SMILES_PATH), sanitize=True)
smiles_dict = {mol.GetProp("_Name"): mol for mol in smi_suppl if mol is not None}

# Pass 1: extract ligands and compute Asphericity to select reference structure
ligand_volumes = {}
for pdb_file in sorted(COMPLEX_DIR.glob("*.pdb")):
    # Pair ligand name with PDB filename; if no match, skip
    lig_name = _match_ligand(pdb_file.stem, smiles_dict)
    if lig_name is None:
        continue
    tmp_pdb = TMP_DIR / f"{lig_name}.pdb"
    u = mda.Universe(pdb_file)
    u.select_atoms(f"chainID {LIGAND_CHAIN_ID}").write(tmp_pdb)
    rdkit_mol = Chem.MolFromPDBFile(str(tmp_pdb), sanitize=True, removeHs=True)
    rdkit_mol = assign_topology(rdkit_mol, smiles_dict[lig_name])
    ligand_volumes[pdb_file] = Asphericity(rdkit_mol)

# Pass 2: align all complexes to reference, extract ligands and reference protein
ref_pdb = max(ligand_volumes, key=ligand_volumes.get)
print(f"Reference structure: {ref_pdb} (Asphericity={ligand_volumes[ref_pdb]:.4f})")
ref_u = mda.Universe(ref_pdb)

for pdb_file in sorted(COMPLEX_DIR.glob("*.pdb")):
    lig_name = _match_ligand(pdb_file.stem, smiles_dict)
    if lig_name is None:
        continue
    mob_u = mda.Universe(pdb_file)
    align.alignto(
        mob_u.select_atoms("protein and backbone"),
        ref_u.select_atoms("protein and backbone"),
    )
    tmp_pdb = TMP_DIR / f"{lig_name}_aligned.pdb"
    mob_u.select_atoms(f"chainID {LIGAND_CHAIN_ID}").write(tmp_pdb)
    rdkit_mol = Chem.MolFromPDBFile(str(tmp_pdb), removeHs=True)
    rdkit_mol = assign_topology(rdkit_mol, smiles_dict[lig_name])
    rdkit_mol.SetProp("_Name", lig_name)
    Chem.MolToMolFile(rdkit_mol, str(ALIGNED_DIR / f"{lig_name}.sdf"))
    print(lig_name)
    view = py3Dmol.view(width=400, height=300)
    view.addModel(Chem.MolToMolBlock(rdkit_mol), "sdf")
    view.setStyle({"stick": {}})
    view.zoomTo()
    view.show()
    if pdb_file == ref_pdb:
        mob_u.select_atoms(f"chainID {PROTEIN_CHAIN_ID}").write(PROTEIN_PATH)

fix_pdb(PROTEIN_PATH, FIXED_PROTEIN_PATH, PH)
view = py3Dmol.view(width=800, height=600)
view.addModel(FIXED_PROTEIN_PATH.read_text(), "pdb")
view.setStyle({"cartoon": {"color": "spectrum"}})
view.zoomTo()
view.show()

## Loading the ligands

We load the pre-aligned ligand SDF files produced in the previous step. Each file has correct bond orders and hydrogens assigned via `assign_topology`.

In [ ]:
suppl = sorted(
    [Chem.MolFromMolFile(str(f), removeHs=False) for f in sorted(ALIGNED_DIR.glob("*.sdf"))],
    key=lambda m: m.GetProp("_Name"),
)

In [ ]:
ligands = [SmallMoleculeComponent(mol, name=mol.GetProp("_Name")) for mol in suppl]

In [ ]:
# Extract the contents of the sdf file and visualise it
ligands_rdmol = list(suppl)

for ligand_rdmol in ligands_rdmol:
    AllChem.Compute2DCoords(ligand_rdmol)

ligand_labels = [mol.GetProp("_Name") if mol.HasProp("_Name") else "" for mol in ligands_rdmol]

Draw.MolsToGridImage(ligands_rdmol, legends=ligand_labels)

## Charging the ligands

It is recommended to use a single set of charges for each ligand to ensure reproducibility between repeats or consistent charges between different legs of a calculation involving the same ligand, like a relative binding affinity calculation for example. 

Here we will use some utility functions from OpenFE which can assign partial charges to a series of molecules with a variety of methods which can be configured via the `OpenFFPartialChargeSettings` class. In this example 
we will charge the ligands using the `am1bcc` method from `ambertools` which is the default charge scheme used by OpenFE.

In [ ]:
charged_sdfs = sorted(CHARGED_DIR.glob("*.sdf"))
if charged_sdfs:
    charged_ligands = [SmallMoleculeComponent.from_sdf_file(str(f)) for f in charged_sdfs]
else:
    charge_settings = OpenFFPartialChargeSettings(
        partial_charge_method="am1bcc", off_toolkit_backend="ambertools"
    )

    charged_ligands = bulk_assign_partial_charges(
        molecules=ligands,
        overwrite=False,
        method=charge_settings.partial_charge_method,
        toolkit_backend=charge_settings.off_toolkit_backend,
        generate_n_conformers=charge_settings.number_of_conformers,
        nagl_model=charge_settings.nagl_model,
        processors=mp.cpu_count(),
    )

    for smc in charged_ligands:
        sdf_path = CHARGED_DIR / f"{smc.name}.sdf"
        sdf_path.write_text(smc.to_sdf())

## Creating the `LigandNetwork`

The first step is to create a `LigandNetwork`, which is a network with small molecules as nodes, and atom mappings, the description of how to alchemically mutate between the molecules, as its edges.

The pipeline for creating a `LigandNetwork` can involve three components:

* **Atom Mapper**: Proposes potential atom mappings (descriptions of the alchemical change) for pairs of ligands.
* **Scorer**: Given an atom mapping, provides an estimate of the quality of that mapping (higher scores are better).
* **Network Planner**: Creates the actual `LigandNetwork`; different network planners provide different strategies.

Each of these components could be replaced by other options.

In [ ]:
lomap_mapper = LomapAtomMapper(
    time=20,  # Time out if MCS algorithm takes 20 seconds
    threed=True,  # Use atom positions to prune symmetric mappings
    max3d=1.0,  # Forbid mapping between atoms more than 1.0 Å apart
    element_change=True,  # Allow mappings that change an atoms element
    seed="",  # Empty SMARTS string causes MCS search to start from scratch
    shift=False,  # Keep pre-aligned atom positions for 3D position checks
)
kartograf_mapper = KartografAtomMapper(atom_map_hydrogens=True)

# Passing multiple atom mappers to the planner
mappers = [lomap_mapper, kartograf_mapper]
scorer = default_lomap_score
network_planner = generate_lomap_network

The exact call signature depends on the network planner: a minimal spanning network requires a score, whereas that is optional for a radial network (but a radial network needs the central ligand to be provided).

You can output the ligand network to the same `graphml` format as we saw in the CLI tutorial with the following:

Now we can look at the overall structure of the `LigandNetwork`:

In [ ]:
ligand_network = network_planner(ligands=charged_ligands, mappers=mappers, scorer=scorer)
plot_atommapping_network(ligand_network)

with LIGAND_NETWORK_PATH.open("w", encoding="utf-8") as f:
    f.write(ligand_network.to_graphml())

We can also inspect the individual atom mappings:

To get the score for this mapping, we inspect its `annotations` attribute. Arbitrary annotations can be added when a mapping is created, although our network generator only includes the score.

In [ ]:
for mapping in ligand_network.edges:
    print(mapping.annotations)
    display(mapping)
    display(mapping.view_3d(show_atomIDs=True))

## Creating a single `Transformation`

The `LigandNetwork` only knows about the small molecules and the alchemical connections between them. It doesn't know anything about environment (e.g., solvent) or about the `Protocol` that will be used during the simulation.

That information in included in a `Transformation`. Each of these transformations corresponds to a single leg of the simulation campaign, so for each edge in the `LigandNetwork`, we will create two `Transformation`s: one for the complex and one for solvent.

In practice, this will be done for each edge of the `LigandNetwork` in a loop, but for illustrative purposes we'll dive into the details of creating a single transformation. In particular, we'll create the solvent leg for the pair of molecules we selecting for the mapping above.

### Creating `ChemicalSystem`s

OpenFE describes complex molecular systems as being composed of `Component`s. For example, we have `SmallMoleculeComponent` for each small molecule in the `LigandNetwork`. We'll create a `SolventComponent` to describe the solvent, and binding free energy calculations involve a `ProteinComponent`.

The `Component`s are joined in a `ChemicalSystem`, which describes all the particles in the simulation.

In [ ]:
# defaults are water with NaCl at 0.15 M
solvent = SolventComponent()

In [ ]:
protein = ProteinComponent.from_pdb_file(str(FIXED_PROTEIN_PATH), name=FIXED_PROTEIN_PATH.stem)

### Creating a `Protocol`

The actual simulation is performed by a `Protocol`. We'll use an OpenMM-based hybrid topology relative free energy `Protocol`.

The easiest way to customize protocol settings is to start with the default settings, and modify them. Many settings carry units with them.

We use the default settings with an adapted solvent padding for the complex phase to avoid adding too many waters.

**Note: Since [OpenFE v1.5.0](https://docs.openfree.energy/en/v1.6.1/CHANGELOG.html#v1-5-0), `real_time_analysis_interval` does not need to be divisible by `checkpoint_interval`.**

**Note: by default the settings use a solvent padding of 1.5 nm, this is appropriate for solvent simulations, but for complex simulations this leads to excess water being in the box, slowing down your simulation. To deal with this, we will create two sets of settings, and two sets of protocols, one for each leg of the transformations.**

In [ ]:
complex_settings = RelativeHybridTopologyProtocol.default_settings()

# Change the value
complex_settings.thermo_settings.temperature = TEMPERATURE * unit.kelvin
complex_settings.thermo_settings.ph = PH
complex_settings.simulation_settings.production_length = PRODUCTION_LENGTH * unit.nanosecond
complex_settings.simulation_settings.n_replicas = LAMBDA_WINDOWS
complex_settings.lambda_settings.lambda_windows = LAMBDA_WINDOWS

# Reduce the solvent padding for the complex phase
complex_settings.solvation_settings.solvent_padding = 1 * unit.nanometer

complex_protocol = RelativeHybridTopologyProtocol(complex_settings)

complex_settings

In [ ]:
solvent_settings = RelativeHybridTopologyProtocol.default_settings()

# Change the value
solvent_settings.thermo_settings.temperature = TEMPERATURE * unit.kelvin
solvent_settings.thermo_settings.ph = PH
solvent_settings.simulation_settings.production_length = PRODUCTION_LENGTH * unit.nanosecond
solvent_settings.simulation_settings.n_replicas = LAMBDA_WINDOWS
solvent_settings.lambda_settings.lambda_windows = LAMBDA_WINDOWS

solvent_protocol = RelativeHybridTopologyProtocol(solvent_settings)

solvent_settings

### Creating the `Transformation`

Once we have the mapping, the two `ChemicalSystem`s, and the `Protocol`, creating the `Transformation` is easy:

To summarize, this `Transformation` contains:
- chemical models of both sides of the alchemical transformation in `systemA` and `systemB`
- the correspondence of items in these two sides in `mapping` 
- a description of the exact computational algorithm to use to perform the estimate in `complex_protocol`

## Creating the `AlchemicalNetwork`

The `AlchemicalNetwork` contains all the information needed to run the entire campaign. It consists of a `Transformation` for each leg of the campaign. We'll loop over all the mappings, and then loop over the legs. In that inner loop, we'll make each transformation.

We can also use the `RBFEAlchemicalNetworkPlanner` to create an `AlchemicalNetwork`.

In [ ]:
transformations = []
for mapping in ligand_network.edges:
    for leg in ["solvent", "complex"]:
        # use the solvent and protein created above
        sysA_dict = {"ligand": mapping.componentA, "solvent": solvent}
        sysB_dict = {"ligand": mapping.componentB, "solvent": solvent}

        if leg == "complex":
            # If this is a complex transformation we use the complex protocol
            # and add in the protein to the chemical states
            protocol = complex_protocol
            sysA_dict["protein"] = protein
            sysB_dict["protein"] = protein
        else:
            # If this is a solvent transformation we just use the solvent protocol
            protocol = solvent_protocol

        # we don't have to name objects, but it can make things (like filenames) more convenient
        sysA = ChemicalSystem(sysA_dict, name=f"{mapping.componentA.name}_{leg}")
        sysB = ChemicalSystem(sysB_dict, name=f"{mapping.componentB.name}_{leg}")

        prefix = "rbfe_"  # prefix is only to exactly reproduce CLI

        transformation = Transformation(
            stateA=sysA,
            stateB=sysB,
            mapping=mapping,
            protocol=protocol,  # use protocol created above
            name=f"{prefix}{sysA.name}_{sysB.name}",
        )
        transformations.append(transformation)

network = AlchemicalNetwork(transformations)

## Writing the `AlchemicalNetwork` to disk

We'll write out each transformation to disk, so that they can be run independently using the `openfe quickrun` command:

In [ ]:
# We write out each transformation
for transformation in network.edges:
    transformation.to_json(TRANSFORMATION_DIR / f"{transformation.name}.json")

# List contents of the transformations directory
os.listdir(TRANSFORMATION_DIR)

Each of these individual `.json` files contains a `Transformation`, which contains all the information to run the calculation.  These could be farmed out as individual jobs on a HPC cluster. These files are identical to what were created in setup stage of the CLI tutorial; for details on running them, follow from the section on running simulations in the CLI tutorial

## SepTop Relative Binding Free Energy Transformations

Unlike the hybrid topology RBFE above, SepTop simulates both the complex and solvent legs in a single run. There is therefore one transformation per network edge (no separate solvent leg).

In [ ]:
septop_settings = SepTopProtocol.default_settings()
septop_settings.thermo_settings.temperature = TEMPERATURE * unit.kelvin
septop_settings.thermo_settings.ph = PH
septop_protocol = SepTopProtocol(septop_settings)

In [ ]:
septop_transformations = []
for mapping in ligand_network.edges:
    ligA, ligB = mapping.componentA, mapping.componentB
    stateA = ChemicalSystem(
        {"ligand": ligA, "protein": protein, "solvent": solvent},
        name=f"{ligA.name}_complex",
    )
    stateB = ChemicalSystem(
        {"ligand": ligB, "protein": protein, "solvent": solvent},
        name=f"{ligB.name}_complex",
    )
    septop_transformations.append(
        Transformation(
            stateA=stateA,
            stateB=stateB,
            mapping=None,
            protocol=septop_protocol,
            name=f"septop_{ligA.name}_{ligB.name}",
        )
    )

for t in septop_transformations:
    t.to_json(SEPTOP_TRANSFORMATION_DIR / f"{t.name}.json")

os.listdir(SEPTOP_TRANSFORMATION_DIR)